# Analyse Exploratoire de Données (EDA) — Marché de l'Emploi Data Science

**Projet DPIA** — Collecte, Stockage et Analyse des Données

Ce notebook réalise l'EDA complète sur les offres d'emploi Data Science collectées depuis :
- **API Adzuna** → stockées en JSONL sur S3/MinIO (`raw/adzuna/YYYY/MM/DD/`)
- **Scraping Welcome to the Jungle** → fichiers JSON/CSV sur S3

**Objectifs :**
1. Charger les données brutes depuis S3 (MinIO en dev)
2. Nettoyer et harmoniser les deux sources
3. Analyser les tendances (salaires, villes, compétences)
4. Visualiser les insights clés
5. Préparer les données pour injection dans PostgreSQL/RDS

## 1. Import des Bibliothèques et Configuration

Chargement de toutes les dépendances nécessaires et configuration de l'environnement d'analyse.
On utilise `boto3` pour S3/MinIO, `pandas` pour la manipulation, `seaborn`/`matplotlib` pour la visualisation.

In [ ]:
# --- Bibliothèques de base ---
import os
import io
import json
import re
import warnings
from collections import Counter
from datetime import datetime

# --- Manipulation de données ---
import pandas as pd
import numpy as np

# --- Connexion S3/MinIO ---
import boto3
from botocore.exceptions import ClientError

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Base de données ---
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# --- Configuration ---
warnings.filterwarnings("ignore")
load_dotenv()

# Affichage Pandas
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

# Style Seaborn
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

# --- Constantes du projet ---
S3_BUCKET = os.getenv("S3_BUCKET_NAME", "dpia-data-bucket")
S3_PREFIX_ADZUNA = os.getenv("S3_PREFIX", "raw/adzuna")
S3_PREFIX_WTJ = "raw/wtj"  # Welcome to the Jungle (scraping)
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_USER = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_PASSWORD = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
ENV_MODE = os.getenv("ENV_MODE", "dev")

print(f"Mode : {ENV_MODE}")
print(f"Bucket : {S3_BUCKET}")
print(f"MinIO  : {MINIO_ENDPOINT}")

## 2. Connexion à S3/MinIO et Chargement des Données

On se connecte au stockage objet (MinIO en dev, AWS S3 en prod) et on charge les fichiers JSONL les plus récents.
Les données Adzuna sont partitionnées par date : `raw/adzuna/YYYY/MM/DD/*.jsonl`

In [ ]:
def create_s3_client():
    """Crée un client S3 adapté au mode (dev=MinIO, prod=AWS)."""
    if ENV_MODE == "dev":
        return boto3.client(
            "s3",
            endpoint_url=MINIO_ENDPOINT,
            aws_access_key_id=MINIO_USER,
            aws_secret_access_key=MINIO_PASSWORD,
        )
    return boto3.client("s3")


def list_s3_files(s3_client, prefix: str) -> list[dict]:
    """Liste tous les fichiers sous un préfixe S3, triés par date (plus récent d'abord)."""
    files = []
    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            files.append({"key": obj["Key"], "modified": obj["LastModified"], "size": obj["Size"]})
    files.sort(key=lambda x: x["modified"], reverse=True)
    return files


def load_jsonl_from_s3(s3_client, key: str) -> list[dict]:
    """Charge un fichier JSONL depuis S3 et retourne une liste de dicts."""
    obj = s3_client.get_object(Bucket=S3_BUCKET, Key=key)
    content = obj["Body"].read().decode("utf-8")
    records = []
    for line in content.strip().split("\n"):
        if line.strip():
            records.append(json.loads(line))
    return records


# --- Connexion ---
s3 = create_s3_client()
print("[OK] Client S3 créé")

# --- Lister les fichiers Adzuna ---
adzuna_files = list_s3_files(s3, S3_PREFIX_ADZUNA)
print(f"\n[Adzuna] {len(adzuna_files)} fichiers trouvés dans '{S3_PREFIX_ADZUNA}/'")
for f in adzuna_files[:5]:
    print(f"  - {f['key']}  ({f['size']} bytes, {f['modified']})")

# --- Lister les fichiers WTJ (si disponibles) ---
try:
    wtj_files = list_s3_files(s3, S3_PREFIX_WTJ)
    print(f"\n[WTJ] {len(wtj_files)} fichiers trouvés dans '{S3_PREFIX_WTJ}/'")
except Exception:
    wtj_files = []
    print("\n[WTJ] Aucun fichier de scraping disponible pour le moment.")

In [ ]:
# --- Charger toutes les données Adzuna dans un DataFrame ---
all_records = []
for f in adzuna_files:
    records = load_jsonl_from_s3(s3, f["key"])
    all_records.extend(records)

print(f"[Adzuna] {len(all_records)} enregistrements chargés au total")

# Chaque enregistrement est enrichi : {source, country, search_query, page, fetched_at, data: {...}}
# On extrait le champ 'data' qui contient l'offre Adzuna brute
df_adzuna = pd.json_normalize([r.get("data", r) for r in all_records])
df_adzuna["_source"] = "adzuna"
df_adzuna["_fetched_at"] = [r.get("fetched_at") for r in all_records]

print(f"[Adzuna] DataFrame : {df_adzuna.shape[0]} lignes x {df_adzuna.shape[1]} colonnes")
df_adzuna.head(3)

In [ ]:
# --- Charger les données WTJ (scraping) si disponibles ---
# Format attendu : JSON ou CSV avec colonnes titre, entreprise, localisation, description, salaire, etc.

if wtj_files:
    wtj_records = []
    for f in wtj_files:
        if f["key"].endswith(".jsonl"):
            wtj_records.extend(load_jsonl_from_s3(s3, f["key"]))
        elif f["key"].endswith(".json"):
            obj = s3.get_object(Bucket=S3_BUCKET, Key=f["key"])
            data = json.loads(obj["Body"].read().decode("utf-8"))
            wtj_records.extend(data if isinstance(data, list) else [data])
        elif f["key"].endswith(".csv"):
            obj = s3.get_object(Bucket=S3_BUCKET, Key=f["key"])
            df_temp = pd.read_csv(io.BytesIO(obj["Body"].read()))
            wtj_records.extend(df_temp.to_dict("records"))

    df_wtj = pd.DataFrame(wtj_records)
    df_wtj["_source"] = "wtj"
    print(f"[WTJ] DataFrame : {df_wtj.shape[0]} lignes x {df_wtj.shape[1]} colonnes")
else:
    df_wtj = pd.DataFrame()
    print("[WTJ] Pas de données WTJ disponibles. On travaille uniquement avec Adzuna.")

## 3. Inspection Initiale des DataFrames

Exploration de la structure brute des données : types, valeurs manquantes, aperçu des premières lignes.
Harmonisation des colonnes entre les deux sources avant concaténation.

In [ ]:
# --- Inspection du DataFrame Adzuna ---
print("=" * 60)
print("ADZUNA — Structure du DataFrame")
print("=" * 60)
print(f"\nShape : {df_adzuna.shape}")
print(f"\nColonnes :\n{list(df_adzuna.columns)}")
print(f"\nTypes :")
print(df_adzuna.dtypes)
print(f"\nAperçu :")
df_adzuna.head(3)

In [ ]:
# --- Harmonisation des colonnes Adzuna ---
# L'API Adzuna retourne des champs imbriqués (location.display_name, company.display_name, category.label)
# json_normalize les a aplatis avec des points (ex: "location.display_name")

# Mapping de renommage pour standardiser
ADZUNA_RENAME = {
    "title": "intitule_poste",
    "description": "description",
    "company.display_name": "entreprise",
    "location.display_name": "ville",
    "location.area": "region_list",
    "category.label": "categorie",
    "contract_type": "type_contrat",
    "contract_time": "temps_contrat",
    "salary_min": "salaire_min",
    "salary_max": "salaire_max",
    "created": "date_publication",
    "redirect_url": "url",
    "latitude": "latitude",
    "longitude": "longitude",
    "id": "id_offre",
    "_source": "source",
    "_fetched_at": "date_collecte",
}

# Renommer uniquement les colonnes qui existent
rename_existing = {k: v for k, v in ADZUNA_RENAME.items() if k in df_adzuna.columns}
df_adzuna_clean = df_adzuna.rename(columns=rename_existing)

# Sélectionner les colonnes utiles
cols_keep = [c for c in ADZUNA_RENAME.values() if c in df_adzuna_clean.columns]
df_adzuna_clean = df_adzuna_clean[cols_keep]

print(f"[Adzuna] Colonnes harmonisées : {list(df_adzuna_clean.columns)}")
df_adzuna_clean.head(3)

In [ ]:
# --- Harmonisation WTJ et concaténation ---
# Si les données WTJ sont disponibles, on harmonise et concatène

if not df_wtj.empty:
    # Adapter le mapping selon les colonnes réelles du scraping WTJ
    WTJ_RENAME = {
        "title": "intitule_poste",
        "company_name": "entreprise",
        "location": "ville",
        "description": "description",
        "contract_type": "type_contrat",
        "salary": "salaire_brut",
        "published_at": "date_publication",
        "url": "url",
        "_source": "source",
    }
    rename_wtj = {k: v for k, v in WTJ_RENAME.items() if k in df_wtj.columns}
    df_wtj_clean = df_wtj.rename(columns=rename_wtj)
    cols_wtj = [c for c in WTJ_RENAME.values() if c in df_wtj_clean.columns]
    df_wtj_clean = df_wtj_clean[cols_wtj]

    # Concaténation des deux sources
    df = pd.concat([df_adzuna_clean, df_wtj_clean], ignore_index=True)
    print(f"[CONCAT] {len(df_adzuna_clean)} Adzuna + {len(df_wtj_clean)} WTJ = {len(df)} total")
else:
    df = df_adzuna_clean.copy()
    print(f"[INFO] Travail sur Adzuna uniquement : {len(df)} offres")

df.info()

## 4. Nettoyage des Données — Valeurs Manquantes

Analyse et traitement des valeurs manquantes. Stratégie :
- Colonnes avec **>70% de NaN** → supprimées
- **Salaires** manquants → médiane par poste
- **Villes** manquantes → "Non précisé"
- **Descriptions** manquantes → chaîne vide

In [ ]:
# --- Visualisation des valeurs manquantes ---
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=True)
missing_pct = missing_pct[missing_pct > 0]

if not missing_pct.empty:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_pct) * 0.4)))
    missing_pct.plot(kind="barh", color="salmon", edgecolor="black", ax=ax)
    ax.set_xlabel("% de valeurs manquantes")
    ax.set_title("Pourcentage de Valeurs Manquantes par Colonne")
    ax.axvline(x=70, color="red", linestyle="--", label="Seuil de suppression (70%)")
    ax.legend()
    plt.tight_layout()
    plt.savefig("missing_values.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("[OK] Aucune valeur manquante détectée !")

In [ ]:
# --- Traitement des valeurs manquantes ---
print(f"Avant nettoyage : {df.shape}")

# 1. Supprimer les colonnes avec > 70% de NaN
threshold = 0.7
cols_to_drop = [col for col in df.columns if df[col].isnull().mean() > threshold]
if cols_to_drop:
    print(f"  Colonnes supprimées (>{threshold*100:.0f}% NaN) : {cols_to_drop}")
    df.drop(columns=cols_to_drop, inplace=True)

# 2. Remplir les salaires manquants par la médiane du poste
for col in ["salaire_min", "salaire_max"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if "intitule_poste" in df.columns:
            df[col] = df.groupby("intitule_poste")[col].transform(
                lambda x: x.fillna(x.median())
            )

# 3. Remplir les villes manquantes
if "ville" in df.columns:
    df["ville"].fillna("Non précisé", inplace=True)

# 4. Remplir les descriptions manquantes
if "description" in df.columns:
    df["description"].fillna("", inplace=True)

# 5. Remplir les types de contrat manquants
if "type_contrat" in df.columns:
    df["type_contrat"].fillna("Non précisé", inplace=True)

print(f"Après nettoyage : {df.shape}")
print(f"\nValeurs manquantes restantes :")
print(df.isnull().sum())

## 5. Suppression des Doublons Inter-Sources

Dédoublonnage en deux étapes :
1. **Doublons exacts** → suppression directe
2. **Doublons quasi-identiques** entre sources (même titre + entreprise + ville normalisés) → on garde l'offre la plus complète

In [ ]:
print(f"Avant dédoublonnage : {len(df)} offres")

# 1. Doublons exacts (toutes colonnes sauf date_collecte)
cols_dedup = [c for c in df.columns if c != "date_collecte"]
n_exact = df.duplicated(subset=cols_dedup).sum()
df.drop_duplicates(subset=cols_dedup, inplace=True)
print(f"  Doublons exacts supprimés : {n_exact}")

# 2. Doublons quasi-identiques (normalisation + comparaison)
# Créer des clés normalisées pour la comparaison
def normalize_text(text):
    """Normalise un texte pour la comparaison : lowercase, strip, suppression accents basique."""
    if pd.isna(text):
        return ""
    return str(text).lower().strip()

# Colonnes de clé de dédoublonnage
key_cols = []
for col in ["intitule_poste", "entreprise", "ville"]:
    if col in df.columns:
        norm_col = f"_norm_{col}"
        df[norm_col] = df[col].apply(normalize_text)
        key_cols.append(norm_col)

if key_cols:
    # Trier par nombre de NaN (moins de NaN = offre plus complète = priorité)
    df["_nan_count"] = df.isnull().sum(axis=1)
    df.sort_values("_nan_count", inplace=True)
    n_before = len(df)
    df.drop_duplicates(subset=key_cols, keep="first", inplace=True)
    n_quasi = n_before - len(df)
    print(f"  Doublons quasi-identiques supprimés : {n_quasi}")

    # Nettoyer les colonnes temporaires
    df.drop(columns=[c for c in df.columns if c.startswith("_norm_")] + ["_nan_count"], inplace=True)

df.reset_index(drop=True, inplace=True)
print(f"\nAprès dédoublonnage : {len(df)} offres uniques")

## 6. Conversion des Dates et Normalisation

- Conversion des dates en `datetime`
- Création de colonnes dérivées (`annee_mois`, `jours_depuis_publication`)
- Calcul du **salaire moyen** à partir de `salaire_min` et `salaire_max`
- Normalisation des villes et types de contrat

In [ ]:
# --- Conversion des dates ---
if "date_publication" in df.columns:
    df["date_publication"] = pd.to_datetime(df["date_publication"], errors="coerce", utc=True)
    df["annee_mois"] = df["date_publication"].dt.to_period("M").astype(str)
    df["jours_depuis_publication"] = (pd.Timestamp.now(tz="UTC") - df["date_publication"]).dt.days

if "date_collecte" in df.columns:
    df["date_collecte"] = pd.to_datetime(df["date_collecte"], errors="coerce", utc=True)

# --- Calcul du salaire moyen ---
if "salaire_min" in df.columns and "salaire_max" in df.columns:
    df["salaire_moyen"] = (df["salaire_min"] + df["salaire_max"]) / 2
    # Salaires annuels uniquement (filtrer les valeurs aberrantes)
    df.loc[df["salaire_moyen"] < 1000, "salaire_moyen"] = np.nan  # Exclure les valeurs < 1000
    df.loc[df["salaire_moyen"] > 200000, "salaire_moyen"] = np.nan  # Exclure les valeurs > 200K

# --- Normalisation des villes ---
if "ville" in df.columns:
    df["ville"] = df["ville"].str.strip().str.title()
    # Regrouper les variantes parisiennes
    df["ville"] = df["ville"].replace({
        "Paris, France": "Paris",
        "Ile-De-France": "Paris",
        "Île-De-France": "Paris",
    })

# --- Normalisation des types de contrat ---
if "type_contrat" in df.columns:
    contract_map = {
        "permanent": "CDI", "cdi": "CDI", "full_time": "CDI",
        "contract": "CDD", "cdd": "CDD",
        "freelance": "Freelance", "independent": "Freelance",
        "internship": "Stage", "stage": "Stage",
        "apprenticeship": "Alternance", "alternance": "Alternance",
        "part_time": "Temps partiel",
    }
    df["type_contrat"] = df["type_contrat"].str.lower().str.strip().replace(contract_map)

print("[OK] Dates converties, salaires calculés, normalisation effectuée")
print(f"\nSalaire moyen global : {df['salaire_moyen'].mean():,.0f} € (médian : {df['salaire_moyen'].median():,.0f} €)" if "salaire_moyen" in df.columns else "")
df[["intitule_poste", "ville", "type_contrat", "salaire_min", "salaire_max", "salaire_moyen", "date_publication"]].head(10)

## 7. Analyse Statistique — Salaires par Poste

Statistiques descriptives des salaires groupées par intitulé de poste.
On ne retient que les postes ayant au moins 5 offres pour assurer la significativité.

In [ ]:
# --- Statistiques salariales par poste ---
if "salaire_moyen" in df.columns and "intitule_poste" in df.columns:
    # Ne garder que les postes avec des données de salaire
    df_salary = df.dropna(subset=["salaire_moyen"])

    salary_stats = (
        df_salary.groupby("intitule_poste")["salaire_moyen"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .rename(columns={
            "count": "nb_offres", "mean": "moyenne", "median": "mediane",
            "std": "ecart_type", "min": "min", "max": "max"
        })
    )

    # Filtrer : au moins 3 offres (ou 5 si assez de données)
    min_offers = 5 if len(salary_stats[salary_stats["nb_offres"] >= 5]) >= 5 else 3
    salary_stats = salary_stats[salary_stats["nb_offres"] >= min_offers]
    salary_stats = salary_stats.sort_values("mediane", ascending=False)

    print(f"Salaire moyen global du marché Data Science : {df_salary['salaire_moyen'].mean():,.0f} €")
    print(f"Salaire médian global : {df_salary['salaire_moyen'].median():,.0f} €")
    print(f"\n{'=' * 60}")
    print(f"Statistiques salariales par poste (>= {min_offers} offres) :")
    print(f"{'=' * 60}")
    salary_stats.round(0)
else:
    print("[INFO] Pas de données de salaire disponibles pour cette analyse.")

## 8. Analyse Statistique — Offres par Ville et Type de Contrat

Répartition géographique et par type de contrat. Tableau croisé ville × contrat pour identifier les tendances.

In [ ]:
# --- Offres par ville (Top 15) ---
if "ville" in df.columns:
    top_villes = df["ville"].value_counts().head(15)
    print("Top 15 des villes avec le plus d'offres Data Science :")
    print(top_villes.to_string())

# --- Répartition par type de contrat ---
if "type_contrat" in df.columns:
    print(f"\n{'=' * 40}")
    print("Répartition par type de contrat :")
    contrat_counts = df["type_contrat"].value_counts()
    contrat_pct = (contrat_counts / len(df) * 100).round(1)
    print(pd.DataFrame({"Nombre": contrat_counts, "%": contrat_pct}))

# --- Tableau croisé ville × type de contrat ---
if "ville" in df.columns and "type_contrat" in df.columns:
    top_10_villes = df["ville"].value_counts().head(10).index
    df_top = df[df["ville"].isin(top_10_villes)]
    crosstab = pd.crosstab(df_top["ville"], df_top["type_contrat"], margins=True)
    print(f"\n{'=' * 40}")
    print("Tableau croisé Ville × Type de contrat (Top 10 villes) :")
    crosstab

# --- Évolution temporelle ---
if "annee_mois" in df.columns:
    print(f"\n{'=' * 40}")
    print("Évolution du nombre d'offres par mois :")
    monthly = df.groupby("annee_mois").size()
    print(monthly.to_string())

## 9. Extraction des Compétences depuis les Descriptions

Recherche de compétences techniques dans les descriptions d'offres via regex.
On utilise des **word boundaries** (`\b`) pour éviter les faux positifs (ex: "R" dans "React").

In [ ]:
# --- Liste des compétences techniques à détecter ---
SKILLS = [
    "Python", "SQL", "R", "Scala", "Java", "Julia",
    "Spark", "PySpark", "Hadoop", "Hive", "Kafka",
    "TensorFlow", "PyTorch", "Scikit-Learn", "Keras", "XGBoost",
    "Pandas", "NumPy", "SciPy",
    "AWS", "Azure", "GCP", "Google Cloud",
    "Docker", "Kubernetes", "Airflow", "dbt", "MLflow",
    "Tableau", "Power BI", "Looker", "Excel",
    "Git", "CI/CD", "Terraform", "Linux",
    "Deep Learning", "Machine Learning", "NLP", "Computer Vision",
    "ETL", "Data Warehouse", "Snowflake", "BigQuery", "Redshift",
    "MongoDB", "PostgreSQL", "MySQL", "Elasticsearch",
    "API", "REST", "FastAPI", "Flask", "Django",
    "Agile", "Scrum",
]


def extract_skills(description: str) -> list[str]:
    """Extrait les compétences mentionnées dans une description d'offre."""
    if not isinstance(description, str) or not description:
        return []
    desc_lower = description.lower()
    found = []
    for skill in SKILLS:
        # Regex avec word boundaries pour les termes courts
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"
        if re.search(pattern, desc_lower):
            found.append(skill)
    return found


# --- Appliquer l'extraction sur toutes les descriptions ---
if "description" in df.columns:
    df["competences"] = df["description"].apply(extract_skills)
    df["nb_competences"] = df["competences"].apply(len)

    # Compteur global de fréquence
    all_skills = []
    for skills_list in df["competences"]:
        all_skills.extend(skills_list)

    skill_counts = Counter(all_skills)
    top_skills = skill_counts.most_common(20)

    print("Top 20 des compétences les plus demandées :")
    print("=" * 45)
    for skill, count in top_skills:
        pct = count / len(df) * 100
        print(f"  {skill:<20s} {count:>4d} offres ({pct:.1f}%)")
else:
    print("[INFO] Colonne 'description' non disponible.")

## 10. Visualisation — Top 10 des Compétences les Plus Demandées

Bar chart horizontal montrant les compétences techniques les plus fréquentes dans les descriptions d'offres Data Science.

In [ ]:
# --- Bar chart : Top 10 compétences ---
if top_skills:
    top_10 = top_skills[:10]
    skills_names = [s[0] for s in top_10][::-1]  # Inverser pour affichage horizontal
    skills_counts = [s[1] for s in top_10][::-1]
    skills_pct = [s / len(df) * 100 for s in skills_counts]

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(skills_names, skills_counts, color=sns.color_palette("viridis", len(skills_names)))

    # Ajouter les % sur chaque barre
    for bar, pct in zip(bars, skills_pct):
        ax.text(
            bar.get_width() + max(skills_counts) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%",
            va="center", fontsize=11, fontweight="bold"
        )

    ax.set_xlabel("Nombre d'offres mentionnant la compétence", fontsize=12)
    ax.set_title("Top 10 des Compétences les Plus Demandées en Data Science", fontsize=14, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig("top10_competences.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("[INFO] Pas de données de compétences à visualiser.")

## 11. Visualisation — Répartition Géographique des Offres

Deux graphiques côte à côte :
- **Bar chart** : Top 15 des villes par nombre d'offres
- **Donut chart** : Répartition par grande région

In [ ]:
# --- Répartition géographique ---
if "ville" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # --- Bar chart : Top 15 villes ---
    top_15_villes = df["ville"].value_counts().head(15)
    colors = sns.color_palette("coolwarm", len(top_15_villes))

    axes[0].bar(range(len(top_15_villes)), top_15_villes.values, color=colors)
    axes[0].set_xticks(range(len(top_15_villes)))
    axes[0].set_xticklabels(top_15_villes.index, rotation=45, ha="right", fontsize=10)
    axes[0].set_ylabel("Nombre d'offres", fontsize=12)
    axes[0].set_title("Top 15 des Villes — Offres Data Science", fontsize=13, fontweight="bold")
    axes[0].spines["top"].set_visible(False)
    axes[0].spines["right"].set_visible(False)

    # Ajouter les valeurs au-dessus des barres
    for i, v in enumerate(top_15_villes.values):
        axes[0].text(i, v + max(top_15_villes.values) * 0.01, str(v), ha="center", fontsize=9, fontweight="bold")

    # --- Donut chart : Répartition par région ---
    # Regrouper les petites villes dans "Autres"
    ville_counts = df["ville"].value_counts()
    top_regions = ville_counts.head(8)
    autres = ville_counts.iloc[8:].sum()
    if autres > 0:
        top_regions = pd.concat([top_regions, pd.Series({"Autres": autres})])

    wedges, texts, autotexts = axes[1].pie(
        top_regions.values,
        labels=top_regions.index,
        autopct="%1.1f%%",
        startangle=90,
        pctdistance=0.8,
        colors=sns.color_palette("Set2", len(top_regions)),
    )
    # Créer le trou central (donut)
    centre = plt.Circle((0, 0), 0.55, fc="white")
    axes[1].add_artist(centre)
    axes[1].set_title("Répartition Géographique des Offres", fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.savefig("repartition_geographique.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("[INFO] Colonne 'ville' non disponible.")

## 12. Visualisation — Boxplot des Salaires par Intitulé de Poste

Distribution des salaires pour les principaux intitulés de poste Data Science.
La ligne verticale pointillée indique le salaire médian global du marché.

In [ ]:
# --- Boxplot : Distribution des salaires par poste ---
if "salaire_moyen" in df.columns and "intitule_poste" in df.columns:
    # Filtrer les postes avec suffisamment de données salariales
    df_box = df.dropna(subset=["salaire_moyen"])
    poste_counts = df_box["intitule_poste"].value_counts()
    postes_valides = poste_counts[poste_counts >= 3].index[:12]  # Top 12 postes

    if len(postes_valides) > 0:
        df_box = df_box[df_box["intitule_poste"].isin(postes_valides)]

        # Trier par salaire médian
        medians = df_box.groupby("intitule_poste")["salaire_moyen"].median().sort_values()
        order = medians.index.tolist()

        fig, ax = plt.subplots(figsize=(14, max(6, len(order) * 0.6)))
        sns.boxplot(
            data=df_box,
            y="intitule_poste",
            x="salaire_moyen",
            order=order,
            palette="YlOrRd",
            ax=ax,
            showfliers=True,
            flierprops=dict(marker="o", markerfacecolor="gray", markersize=4, alpha=0.5),
        )

        # Ligne verticale : salaire médian global
        global_median = df_box["salaire_moyen"].median()
        ax.axvline(
            x=global_median, color="navy", linestyle="--", linewidth=1.5,
            label=f"Médiane globale : {global_median:,.0f} €"
        )

        ax.set_xlabel("Salaire annuel moyen (€)", fontsize=12)
        ax.set_ylabel("")
        ax.set_title("Distribution des Salaires par Intitulé de Poste", fontsize=14, fontweight="bold")
        ax.legend(fontsize=11)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()
        plt.savefig("boxplot_salaires.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("[INFO] Pas assez de données salariales par poste pour le boxplot.")
else:
    print("[INFO] Colonnes 'salaire_moyen' ou 'intitule_poste' non disponibles.")

## 13. Préparation du DataFrame pour Injection dans PostgreSQL / AWS RDS

Transformation du DataFrame nettoyé en un format prêt pour insertion dans la base de données.
- En **dev** : PostgreSQL local (Docker)
- En **prod** : AWS RDS PostgreSQL

> **Sécurité** : Les credentials sont lus depuis les variables d'environnement (`.env`), jamais codés en dur.

In [ ]:
# --- Préparer le DataFrame final pour insertion SQL ---

# Colonnes à injecter dans la table SQL
SQL_COLUMNS = [
    "intitule_poste", "entreprise", "ville", "type_contrat",
    "salaire_min", "salaire_max", "salaire_moyen",
    "description", "categorie", "date_publication", "source", "url",
]

# Ne garder que les colonnes disponibles
cols_sql = [c for c in SQL_COLUMNS if c in df.columns]
df_sql = df[cols_sql].copy()

# Convertir les dates en string pour compatibilité SQL
if "date_publication" in df_sql.columns:
    df_sql["date_publication"] = df_sql["date_publication"].astype(str)

# Tronquer les descriptions trop longues (TEXT dans PG mais on limite pour la preview)
if "description" in df_sql.columns:
    df_sql["description"] = df_sql["description"].str[:5000]

print(f"DataFrame prêt pour SQL : {df_sql.shape[0]} lignes x {df_sql.shape[1]} colonnes")
print(f"\nColonnes : {list(df_sql.columns)}")
print(f"\nTypes :")
print(df_sql.dtypes)
df_sql.head(3)

In [ ]:
# --- Injection dans PostgreSQL (dev) ou AWS RDS (prod) ---
# Les credentials sont lus depuis les variables d'environnement (.env)

if ENV_MODE == "dev":
    # Dev : PostgreSQL local dans Docker Compose
    DB_HOST = os.getenv("RDS_HOST", "postgres")  # nom du service Docker
    DB_PORT = os.getenv("RDS_PORT", "5432")
    DB_NAME = os.getenv("RDS_DB_NAME", "dpia_db")
    DB_USER = os.getenv("RDS_USERNAME", "airflow")
    DB_PASS = os.getenv("RDS_PASSWORD", "airflow")
else:
    # Prod : AWS RDS PostgreSQL
    DB_HOST = os.getenv("RDS_HOST")
    DB_PORT = os.getenv("RDS_PORT", "5432")
    DB_NAME = os.getenv("RDS_DB_NAME")
    DB_USER = os.getenv("RDS_USERNAME")
    DB_PASS = os.getenv("RDS_PASSWORD")

TABLE_NAME = "offres_emploi_data_science"

try:
    # Connexion via SQLAlchemy
    engine = create_engine(
        f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    )

    # Insertion avec gestion des chunks pour les gros volumes
    df_sql.to_sql(
        TABLE_NAME,
        engine,
        if_exists="replace",
        index=False,
        chunksize=500,
        method="multi",
    )
    print(f"[OK] {len(df_sql)} lignes insérées dans la table '{TABLE_NAME}'")

    # Vérification : lire les premières lignes
    with engine.connect() as conn:
        result = pd.read_sql(text(f"SELECT COUNT(*) as total FROM {TABLE_NAME}"), conn)
        print(f"[CHECK] {result['total'].iloc[0]} lignes dans la table")

        preview = pd.read_sql(text(f"SELECT * FROM {TABLE_NAME} LIMIT 5"), conn)
    preview

except Exception as e:
    print(f"[ERREUR] Connexion ou insertion échouée : {e}")
    print("Vérifiez que PostgreSQL est démarré et que les credentials dans .env sont corrects.")

## Résumé de l'EDA

| Étape | Résultat |
|-------|---------|
| **Sources** | Adzuna (API) + Welcome to the Jungle (Scraping) |
| **Stockage brut** | S3/MinIO — JSONL partitionné par date |
| **Nettoyage** | Valeurs manquantes traitées, doublons inter-sources supprimés |
| **Compétences** | Extraction regex depuis les descriptions |
| **Visualisations** | Top 10 compétences, répartition géo, boxplot salaires |
| **Sortie** | DataFrame propre injecté dans PostgreSQL (`offres_emploi_data_science`) |

**Fichiers exportés :**
- `missing_values.png`
- `top10_competences.png`
- `repartition_geographique.png`
- `boxplot_salaires.png`

**Prochaines étapes :**
- Enrichir avec le scraping WTJ une fois les données collectées
- Créer le dashboard interactif (Plotly / Grafana)
- Automatiser l'EDA via un DAG Airflow